# Pipeline NLP : comédie ou horreur

## Partie 1 — Audit et nettoyage

Objectifs de cette partie :

- charger le CSV source sans le modifier ;
- vérifier son schéma, les valeurs manquantes et les classes ;
- mesurer la longueur des résumés ;
- repérer les doublons exacts et les éventuels conflits d'étiquettes ;
- créer en mémoire un jeu nettoyé avant tout découpage.

Convention : `0 = comédie` et `1 = horreur`.

### 1.1 Imports et chemin du fichier

`Path` construit des chemins compatibles avec Windows. Le bloc ci-dessous fonctionne lorsque le notebook est lancé depuis la racine du projet ou depuis le dossier `notebooks`. Une assertion arrête immédiatement l'exécution si le CSV est introuvable.

In [2]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "textes_films_horreur_comedie.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "textes_films_horreur_comedie.csv"
assert DATA_PATH.exists(), f"CSV introuvable : {DATA_PATH}"

print(f"Racine du projet : {PROJECT_ROOT}")
print(f"Fichier source    : {DATA_PATH}")

Racine du projet : C:\Users\driou\OneDrive\Bureau\modeleIA
Fichier source    : C:\Users\driou\OneDrive\Bureau\modeleIA\textes_films_horreur_comedie.csv


### 1.2 Chargement et schéma

Une ligne de `DataFrame` représente un film. Les colonnes attendues sont le titre, le résumé et la cible. Nous recalculons les dimensions à chaque exécution au lieu de les saisir comme résultats fixes.

In [3]:
df_raw = pd.read_csv(DATA_PATH)
EXPECTED_COLUMNS = ["titre", "resume", "cible"]

print(f"Dimensions : {df_raw.shape[0]} lignes × {df_raw.shape[1]} colonnes")
display(df_raw.head(3))
display(df_raw.dtypes.rename("type pandas").to_frame())

assert list(df_raw.columns) == EXPECTED_COLUMNS, (
    f"Colonnes inattendues : {list(df_raw.columns)}"
)

Dimensions : 2400 lignes × 3 colonnes


,titre,resume,cible
0,Chiedo asilo,"Roberto has a new job as the teacher at a nursery school. The first child he meets is Gianluigi, who's mute. Roberto...",0
1,Friday the 13th: A New Beginning,"Five years after killing the goalie hockey-masked killer Jason Voorhees, Tommy Jarvis has grown up in various mental...",1
2,Stranger Than Fiction,Everybody knows that your life is a story. But what if a story was your life? Harold Crick is your average IRS agent...,0


,type pandas
titre,str
resume,str
cible,int64


### 1.3 Valeurs manquantes, textes vides et classes

Une valeur manquante est représentée par `NaN`. Un texte vide peut ne pas être manquant : par exemple, une chaîne ne contenant que des espaces. Nous contrôlons donc séparément ces deux cas.

In [4]:
missing_counts = df_raw.isna().sum()
empty_counts = pd.Series(
    {
        column: df_raw[column].fillna("").astype(str).str.strip().eq("").sum()
        for column in ["titre", "resume"]
    },
    name="textes vides",
)

display(missing_counts.rename("valeurs manquantes").to_frame())
display(empty_counts.to_frame())

class_counts = df_raw["cible"].value_counts().sort_index()
class_proportions = df_raw["cible"].value_counts(normalize=True).sort_index()
class_summary = pd.DataFrame(
    {"effectif": class_counts, "proportion": class_proportions}
)
display(class_summary)

,valeurs manquantes
titre,0
resume,0
cible,0


,textes vides
titre,0
resume,0


,effectif,proportion
cible,,
0,1200,0.5
1,1200,0.5


### 1.4 Longueur des résumés

La longueur en caractères donne une indication de stockage. La longueur en mots aide à comprendre la quantité de texte disponible par observation. `describe()` fournit notamment le minimum, la médiane (`50%`) et le maximum.

In [5]:
resume_lengths = pd.DataFrame(
    {
        "caracteres": df_raw["resume"].str.len(),
        "mots": df_raw["resume"].str.split().str.len(),
    }
)
display(resume_lengths.describe().round(2))

,caracteres,mots
count,2400.00,2400.00
mean,558.19,96.70
std,349.28,61.26
min,201.00,30.00
25%,335.00,59.00
50%,474.00,82.00
75%,679.00,117.00
max,7104.00,1287.00


### 1.5 Doublons exacts

`duplicated(keep=False)` sélectionne toutes les lignes appartenant à un groupe répété. `duplicated(keep="first")` compte seulement les occurrences excédentaires à supprimer : la première occurrence de chaque groupe est conservée.

In [6]:
exact_duplicate_mask = df_raw.duplicated(
    subset=EXPECTED_COLUMNS, keep=False
)
duplicate_excess_count = int(
    df_raw.duplicated(subset=EXPECTED_COLUMNS, keep="first").sum()
)
duplicate_groups = (
    df_raw.loc[exact_duplicate_mask]
    .value_counts(subset=EXPECTED_COLUMNS)
    .rename("taille_groupe")
    .reset_index()
)

print(f"Lignes appartenant à un groupe répété : {exact_duplicate_mask.sum()}")
print(f"Groupes de doublons exacts              : {len(duplicate_groups)}")
print(f"Occurrences excédentaires à supprimer   : {duplicate_excess_count}")
display(duplicate_groups.head(5))

Lignes appartenant à un groupe répété : 105
Groupes de doublons exacts              : 51
Occurrences excédentaires à supprimer   : 54


,titre,resume,cible,taille_groupe
0,The Beast,Fishing and jobs are becoming scarce in a small seaport community when tragedy strikes a young couple disappear into...,1,3
1,The Bees,Corporate smuggling of South American killer bees into the United States results in huge swarms terrorizing the nort...,1,3
2,Survival Island,"Jack & Jennifer take a private cruise with friends, such as Captain Richards and his wife Maggie. Manuel is a staff ...",1,3
3,Grave Secrets: The Legacy of Hilltop Drive,Jean and Shag Williams locate a newly built house and decide it's perfect for them to buy. One thing the developers ...,1,2
4,Horror Planet,"A group of scientists are working in a lab located on a distant planet. One of them, Sandy, is attacked, raped and i...",1,2


### 1.6 Résumés identiques avec des étiquettes contradictoires

Un même résumé associé à `0` et à `1` rendrait la vérité terrain contradictoire. `nunique()` compte le nombre de cibles différentes pour chaque résumé.

In [7]:
label_count_per_resume = df_raw.groupby("resume")["cible"].nunique()
conflicting_resumes = label_count_per_resume[label_count_per_resume > 1]

print(f"Résumés associés à plusieurs étiquettes : {len(conflicting_resumes)}")
if not conflicting_resumes.empty:
    display(
        df_raw[df_raw["resume"].isin(conflicting_resumes.index)]
        .sort_values(["resume", "cible"])
    )

Résumés associés à plusieurs étiquettes : 0


### 1.7 Nettoyage en mémoire et contrôles

Nous conservons la première occurrence de chaque ligne exacte. Cette opération est faite **avant** le découpage : sinon deux copies d'un même film pourraient se retrouver dans des lots différents et produire une fuite de données. Le fichier CSV original n'est pas réécrit.

In [8]:
df_clean = (
    df_raw.drop_duplicates(subset=EXPECTED_COLUMNS, keep="first")
    .reset_index(drop=True)
    .copy()
)

print(f"Avant nettoyage : {len(df_raw)} lignes")
print(f"Après nettoyage : {len(df_clean)} lignes")
print(f"Lignes retirées  : {len(df_raw) - len(df_clean)}")
display(df_clean["cible"].value_counts().sort_index().rename("effectif").to_frame())

assert df_clean.duplicated(subset=EXPECTED_COLUMNS).sum() == 0
assert df_clean[EXPECTED_COLUMNS].isna().sum().sum() == 0
assert set(df_clean["cible"].unique()) == {0, 1}
assert conflicting_resumes.empty
assert len(df_raw) - len(df_clean) == duplicate_excess_count

print("Contrôles de nettoyage : OK")

Avant nettoyage : 2400 lignes
Après nettoyage : 2346 lignes
Lignes retirées  : 54


,effectif
cible,
0,1195
1,1151


Contrôles de nettoyage : OK


### Résultats attendus avec le fichier fourni

L'exécution doit retrouver 2 400 lignes avant nettoyage, 54 occurrences excédentaires et 2 346 lignes après nettoyage. Aucun résumé ne doit porter deux étiquettes différentes et la dernière cellule doit afficher `Contrôles de nettoyage : OK`.

Questions de compréhension :

1. Pourquoi faut-il retirer les doublons avant de constituer les différents lots ?
2. Quelle différence y a-t-il entre une valeur manquante et une chaîne composée uniquement d'espaces ?

## Partie 2 — Découpage reproductible des données

Objectifs :

- créer un réservoir d'apprentissage de 60 %, un test final de 20 % et un lot de production de 20 % ;
- conserver approximativement la proportion des classes avec la stratification ;
- créer la validation interne uniquement à partir du réservoir d'apprentissage ;
- conserver des identifiants permettant de reconstruire exactement les lots.

Le test final ne servira qu'à l'unique campagne d'évaluation finale. Le lot de production ne servira ni à l'apprentissage, ni au réglage, ni au calcul du score final.

### 2.1 Découpage principal 60/20/20

`random_state` rend le tirage reproductible. `stratify=df["cible"]` demande à scikit-learn de préserver la distribution de la cible. Nous séparons d'abord 60 % et 40 %, puis divisons les 40 % restants en deux moitiés. Comme 2 346 n'est pas divisible exactement selon ces proportions, un écart d'une ligne dû aux arrondis est normal.

In [9]:
from sklearn.model_selection import train_test_split

SPLIT_SEED = 42

# Identifiant stable dans le jeu nettoyé : il permettra de reconstruire les lots.
df_with_id = df_clean.reset_index(names="clean_row_id")

train_pool, temporary_holdout = train_test_split(
    df_with_id,
    test_size=0.40,
    random_state=SPLIT_SEED,
    stratify=df_with_id["cible"],
)
test_final, production_holdout = train_test_split(
    temporary_holdout,
    test_size=0.50,
    random_state=SPLIT_SEED,
    stratify=temporary_holdout["cible"],
)

# La variable intermédiaire n'est plus utile après la création des deux lots.
del temporary_holdout

### 2.2 Validation interne dans les seuls 60 % d'apprentissage

Nous affectons 20 % du réservoir d'apprentissage à la validation interne. Cela représente environ `0,20 × 0,60 = 0,12`, donc 12 % du jeu complet. Les modèles apprendront d'abord sur environ 48 % des données et seront comparés sur ces 12 %. Une fois les choix figés, le réservoir complet de 60 % pourra être utilisé pour le réentraînement final.

In [10]:
train_internal, validation_internal = train_test_split(
    train_pool,
    test_size=0.20,
    random_state=SPLIT_SEED,
    stratify=train_pool["cible"],
)

### 2.3 Effectifs et absence de chevauchement

Une fuite de données apparaîtrait si le même identifiant se trouvait dans plusieurs lots principaux. Les assertions suivantes vérifient que les lots sont disjoints, qu'aucune ligne n'a été perdue et que la validation interne provient uniquement du réservoir d'apprentissage.

In [11]:
def summarize_splits(split_mapping, total_rows):
    rows = []
    for split_name, frame in split_mapping.items():
        counts = frame["cible"].value_counts().reindex([0, 1], fill_value=0)
        rows.append(
            {
                "lot": split_name,
                "lignes": len(frame),
                "part_du_total": len(frame) / total_rows,
                "comedie_0": int(counts.loc[0]),
                "horreur_1": int(counts.loc[1]),
                "proportion_horreur": frame["cible"].mean(),
            }
        )
    return pd.DataFrame(rows).set_index("lot")

main_splits = {
    "apprentissage_60": train_pool,
    "test_final_20": test_final,
    "production_20": production_holdout,
}
internal_splits = {
    "apprentissage_interne": train_internal,
    "validation_interne": validation_internal,
}

display(summarize_splits(main_splits, len(df_with_id)).round(4))
display(summarize_splits(internal_splits, len(df_with_id)).round(4))

main_id_sets = {
    name: set(frame["clean_row_id"]) for name, frame in main_splits.items()
}
all_clean_ids = set(df_with_id["clean_row_id"])

assert main_id_sets["apprentissage_60"].isdisjoint(main_id_sets["test_final_20"])
assert main_id_sets["apprentissage_60"].isdisjoint(main_id_sets["production_20"])
assert main_id_sets["test_final_20"].isdisjoint(main_id_sets["production_20"])
assert set().union(*main_id_sets.values()) == all_clean_ids

train_internal_ids = set(train_internal["clean_row_id"])
validation_internal_ids = set(validation_internal["clean_row_id"])
assert train_internal_ids.isdisjoint(validation_internal_ids)
assert train_internal_ids | validation_internal_ids == main_id_sets["apprentissage_60"]

print("Contrôles des découpages : OK")

,lignes,part_du_total,comedie_0,horreur_1,proportion_horreur
lot,,,,,
apprentissage_60,1407,0.5997,717,690,0.4904
test_final_20,469,0.1999,239,230,0.4904
production_20,470,0.2003,239,231,0.4915


,lignes,part_du_total,comedie_0,horreur_1,proportion_horreur
lot,,,,,
apprentissage_interne,1125,0.4795,573,552,0.4907
validation_interne,282,0.1202,144,138,0.4894


Contrôles des découpages : OK


### 2.4 Conservation des indices

Nous enregistrons uniquement les identifiants des lignes, la graine et l'empreinte SHA-256 du CSV — pas une nouvelle copie des textes. L'empreinte permet de détecter si le fichier source a changé. Le JSON rendra le découpage vérifiable hors du notebook.

In [12]:
import hashlib
import json

SPLIT_INDEX_PATH = PROJECT_ROOT / "data" / "splits" / "split_indices.json"
SPLIT_INDEX_PATH.parent.mkdir(parents=True, exist_ok=True)

with DATA_PATH.open("rb") as source_file:
    source_sha256 = hashlib.file_digest(source_file, "sha256").hexdigest()

split_payload = {
    "schema_version": 1,
    "source_file": DATA_PATH.name,
    "source_sha256": source_sha256,
    "split_seed": SPLIT_SEED,
    "internal_validation_fraction_of_train_pool": 0.20,
    "splits": {
        "train_internal": sorted(train_internal_ids),
        "validation_internal": sorted(validation_internal_ids),
        "test_final": sorted(main_id_sets["test_final_20"]),
        "production_holdout": sorted(main_id_sets["production_20"]),
    },
}

with SPLIT_INDEX_PATH.open("w", encoding="utf-8") as output_file:
    json.dump(split_payload, output_file, ensure_ascii=False, indent=2)

print(f"Indices enregistrés : {SPLIT_INDEX_PATH}")
print(f"SHA-256 du CSV       : {source_sha256}")

Indices enregistrés : C:\Users\driou\OneDrive\Bureau\modeleIA\data\splits\split_indices.json
SHA-256 du CSV       : 8f23e70f02dee0abba114cd5988ea20a505b611a1637d027177f5c87fe7cdb19


### Contrôle de fin de partie 2

La cellule de contrôle doit afficher `Contrôles des découpages : OK`. À partir de maintenant, seules `train_internal` et `validation_internal` peuvent servir aux comparaisons et aux réglages.

Questions de compréhension :

1. Pourquoi le second découpage principal utilise-t-il `test_size=0.50` alors que nous voulons 20 % pour chaque lot ?
2. Quelle serait la fuite méthodologique si nous choisissions un hyperparamètre après avoir regardé son score sur `test_final` ?

## Partie 3 — Représentation TF-IDF sans fuite de données

Un algorithme de machine learning ne reçoit pas directement des phrases : il reçoit une matrice numérique. TF-IDF attribue un poids élevé aux termes fréquents dans un document mais relativement rares dans l'ensemble d'apprentissage.

Pour un terme `t` et un document `d` :

- avec `sublinear_tf=True` : `TF(t,d) = 1 + ln(nombre d'occurrences)` lorsque le terme est présent ;
- avec le lissage de scikit-learn : `IDF(t) = ln((1+n)/(1+df(t))) + 1` ;
- avant normalisation : `TF-IDF(t,d) = TF(t,d) × IDF(t)`.

`n` est le nombre de documents d'apprentissage et `df(t)` le nombre de ces documents contenant le terme. Exemple : pour 3 documents et un terme présent dans un seul, `IDF = ln(4/2)+1 ≈ 1,693`. S'il apparaît deux fois dans un document, son TF sous-linéaire vaut aussi `1+ln(2) ≈ 1,693`, donc son poids brut vaut environ `2,867`.

### 3.1 Paramètres du vectoriseur

Les choix ci-dessous constituent un prétraitement de départ commun aux modèles :

- minuscules et suppression des accents Unicode ;
- unigrammes et bigrammes, par exemple `film` et `film drôle` ;
- termes présents dans au moins deux documents et dans au plus 95 % des documents ;
- maximum de 10 000 caractéristiques pour maîtriser la mémoire ;
- normalisation L2 : la racine de la somme des poids au carré vaut 1 pour chaque ligne non vide ;
- nombres `float32`, suffisants ici et deux fois moins volumineux que `float64`.

Nous n'appliquons pas de stemming ou de lemmatisation à ce stade : TF-IDF fournit une référence simple, reproductible et sérialisable sans dépendance linguistique supplémentaire.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

TFIDF_PARAMS = {
    "lowercase": True,
    "strip_accents": "unicode",
    "ngram_range": (1, 2),
    "min_df": 2,
    "max_df": 0.95,
    "max_features": 10_000,
    "sublinear_tf": True,
    "norm": "l2",
    "dtype": np.float32,
}

tfidf_vectorizer = TfidfVectorizer(**TFIDF_PARAMS)

# fit_transform apprend le vocabulaire et l'IDF sur l'apprentissage autorisé.
X_train_internal = tfidf_vectorizer.fit_transform(train_internal["resume"])

# transform réutilise strictement ce qui vient d'être appris, sans réajustement.
X_validation_internal = tfidf_vectorizer.transform(validation_internal["resume"])

y_train_internal = train_internal["cible"].to_numpy(dtype=np.int32)
y_validation_internal = validation_internal["cible"].to_numpy(dtype=np.int32)

### 3.2 Vocabulaire et dimensions

Dans une matrice TF-IDF, une ligne correspond à un résumé et une colonne à une caractéristique du vocabulaire. Les matrices d'apprentissage et de validation ont donc des nombres de lignes différents, mais exactement les mêmes colonnes. Un terme rencontré uniquement en validation est ignoré puisqu'il ne possède pas de colonne apprise.

In [ ]:
feature_names = tfidf_vectorizer.get_feature_names_out()

print(f"Matrice d'apprentissage : {X_train_internal.shape}")
print(f"Matrice de validation    : {X_validation_internal.shape}")
print(f"Taille du vocabulaire   : {len(feature_names)}")
print(f"Type de matrice         : {type(X_train_internal).__name__}")
print(f"Type numérique          : {X_train_internal.dtype}")

assert X_train_internal.shape[0] == len(train_internal)
assert X_validation_internal.shape[0] == len(validation_internal)
assert X_train_internal.shape[1] == X_validation_internal.shape[1]
assert X_train_internal.shape[1] == len(feature_names)
assert tfidf_vectorizer.idf_.shape[0] == len(feature_names)

row_norms = np.sqrt(X_train_internal.multiply(X_train_internal).sum(axis=1)).A1
assert np.allclose(row_norms[row_norms > 0], 1.0, atol=1e-5)
print("Contrôles TF-IDF : OK")

### 3.3 Pourquoi une matrice creuse ?

Un résumé n'utilise qu'une petite partie du vocabulaire. Une matrice dense mémoriserait aussi tous les zéros. Le format CSR ne stocke essentiellement que les valeurs non nulles, leurs colonnes et la position de début de chaque ligne. Nous calculons ci-dessous le coût réel de la matrice creuse et le coût théorique de sa version dense, sans effectuer la conversion.

In [ ]:
def sparse_size_mib(matrix):
    byte_count = matrix.data.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes
    return byte_count / (1024**2)

def dense_size_mib(matrix):
    byte_count = matrix.shape[0] * matrix.shape[1] * matrix.dtype.itemsize
    return byte_count / (1024**2)

non_zero_count = X_train_internal.nnz
total_cell_count = X_train_internal.shape[0] * X_train_internal.shape[1]
density = non_zero_count / total_cell_count

print(f"Valeurs non nulles       : {non_zero_count:,}")
print(f"Densité                  : {density:.4%}")
print(f"Mémoire CSR mesurée      : {sparse_size_mib(X_train_internal):.2f} Mio")
print(f"Mémoire dense théorique  : {dense_size_mib(X_train_internal):.2f} Mio")

### 3.4 Exemple des poids d'un résumé

Les poids ci-dessous ne représentent pas encore l'importance pour une classe. Ils indiquent seulement quelles caractéristiques TF-IDF sont les plus présentes dans ce résumé relativement au corpus d'apprentissage.

In [ ]:
example_row = X_train_internal[0]
top_positions = np.argsort(example_row.data)[-10:][::-1]
top_feature_indices = example_row.indices[top_positions]

example_weights = pd.DataFrame(
    {
        "caracteristique": feature_names[top_feature_indices],
        "poids_tfidf": example_row.data[top_positions],
    }
)
display(train_internal.iloc[[0]][["titre", "cible", "resume"]])
display(example_weights)

### Contrôle de fin de partie 3

La cellule de dimensions doit afficher `Contrôles TF-IDF : OK`. Nous n'avons créé volontairement aucune matrice pour `test_final` ou `production_holdout` : ces lots restent réservés.

Questions de compréhension :

1. Pourquoi utilisons-nous `fit_transform` sur l'apprentissage mais seulement `transform` sur la validation ?
2. Pourquoi la conversion immédiate de toute la matrice TF-IDF en tableau dense est-elle généralement une mauvaise idée ?